# Titanic Survival Prediction - Improved Workflow
This notebook demonstrates a robust workflow for predicting Titanic survival using XGBoost and scikit-learn best practices. Steps include data cleaning, feature engineering, model selection, evaluation, and submission file creation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
pd.set_option('display.max_colwidth', None)

In [ ]:
df = pd.read_csv('./datasets/train.csv')
test_df = pd.read_csv('./datasets/test.csv')
submission_df = pd.read_csv('./datasets/gender_submission.csv')

In [ ]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Data Cleaning & Feature Engineering
def preprocess(df):
    df = df.copy()
    df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    df = pd.get_dummies(df, columns=['Embarked', 'Sex'], drop_first=True)
    return df

df = preprocess(df)
test_df = preprocess(test_df)
df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Sex_male
0,1,0,3,22.0,1,0,7.2500,False,True,True
1,2,1,1,38.0,1,0,71.2833,False,False,False
2,3,1,3,26.0,0,0,7.9250,False,True,False
3,4,1,1,35.0,1,0,53.1000,False,True,False
4,5,0,3,35.0,0,0,8.0500,False,True,True


In [ ]:
# Split Data
y = df['Survived']
X = df.drop(['Survived', 'PassengerId'], axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, stratify=y, random_state=42)

In [ ]:
# Hyperparameter Tuning with GridSearchCV
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'alpha': [1, 10, 100]
}
xgb = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')
grid = GridSearchCV(xgb, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
print('Best Parameters:', grid.best_params_)
print('Best CV Accuracy:', grid.best_score_)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Best Parameters: {'alpha': 1, 'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 100}
Best CV Accuracy: 0.8272583294921344


In [ ]:
print(accuracy)
best_accuracy = max(accuracy)
best_index = accuracy.index(best_accuracy)
# Unravel the best_index to get the parameter indices for each hyperparameter
import numpy as np
param_grid_shape = (
    len(values['alpha']),
    len(values['learning_rate']),
    len(values['max_depth']),
    len(values['n_estimators'])
    )
alpha_idx, lr_idx, md_idx, ne_idx = np.unravel_index(best_index, param_grid_shape)
best_max_depth = values['max_depth'][md_idx]
best_learning_rate = values['learning_rate'][lr_idx]
best_n_estimators = values['n_estimators'][ne_idx]
best_alpha = values['alpha'][alpha_idx]

params = {
    'objective':'binary:logistic',
    'max_depth':best_max_depth,
    'learning_rate':best_learning_rate,
    'n_estimators':best_n_estimators,
    'alpha':best_alpha
}

[0.7972027972027972, 0.7972027972027972, 0.7972027972027972, 0.7972027972027972, 0.7902097902097902, 0.8041958041958042, 0.7902097902097902, 0.7832167832167832, 0.7902097902097902, 0.7972027972027972, 0.8111888111888111, 0.8041958041958042, 0.7902097902097902, 0.7832167832167832, 0.7832167832167832, 0.7902097902097902, 0.7972027972027972, 0.7832167832167832, 0.8321678321678322, 0.8111888111888111, 0.8111888111888111, 0.7832167832167832, 0.7972027972027972, 0.7832167832167832, 0.7972027972027972, 0.7762237762237763, 0.7762237762237763, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832, 0.7832167832167832,

In [ ]:
# Train Final Model and Evaluate
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_val)
print('Validation Accuracy:', accuracy_score(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

Validation Accuracy: 0.8044692737430168
[[105   5]
 [ 30  39]]
              precision    recall  f1-score   support

           0       0.78      0.95      0.86       110
           1       0.89      0.57      0.69        69

    accuracy                           0.80       179
   macro avg       0.83      0.76      0.77       179
weighted avg       0.82      0.80      0.79       179



In [ ]:
# Prepare Test Data for Submission
X_test = test_df.drop(['PassengerId'], axis=1)
test_pred = best_model.predict(X_test)
submission_df['Survived'] = test_pred
submission_df.to_csv('./datasets/my_submission.csv', index=False)  # Save to a new file

In [ ]:
test_df.shape

(418, 9)

In [ ]:
X_test = test_df.drop(['PassengerId'], axis=1)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
y_pred = best_model.predict(X_test)
submission_df["Survived"] = y_pred
submission_df.to_csv('./datasets/submission.csv', index=False)  # Save the results to CSV